In [1]:
import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
workers = 2
batch_size = 10
nz = 100
num_epochs = 5
lr = 0.001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

fsc_test={}
for i in os.listdir('/kaggle/input/cidaut-ai-fake-scene-classification-2024/Test'):
    fsc_test[i]=TF.to_tensor(cv2.resize(cv2.imread(f'/kaggle/input/cidaut-ai-fake-scene-classification-2024/Test/{i}'),(256, 455)))

print(fsc_test["21.jpg"].shape)
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

torch.Size([3, 455, 256])


In [5]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class FSC(nn.Module):
    def __init__(self):
        super().__init__()
        self.FSC_rafire=nn.Sequential(
            nn.Conv2d(3, 4,3),
            nn.BatchNorm2d(4),
            nn.LeakyReLU(),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(4, 8, 3),
            nn.BatchNorm2d(8),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(8, 16, 3),
            nn.BatchNorm2d(16),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(16, 32, 3),
            nn.BatchNorm2d(32),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(64, 128, 3),
            nn.BatchNorm2d(128),
            nn.AvgPool2d(2, 2),

            nn.Flatten(),

            nn.Linear(1280, 264),
            nn.LeakyReLU(),
            nn.Linear(264, 132),
            nn.LeakyReLU(),
            nn.Linear(132, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 2)
        )

    def forward(self,x):
        return self.FSC_rafire(x)

In [6]:
FSC_net = FSC().to(device)
if (device.type == 'cuda') and (ngpu > 1):
    FSC_net = nn.DataParallel(FSC_net, list(range(ngpu)))
FSC_net.apply(weights_init)

FSC_net.load_state_dict(torch.load(f"/kaggle/input/fake-scene-classification-model/670.pth",weights_only=False))


<All keys matched successfully>

In [7]:

criterion,img_devil = nn.CrossEntropyLoss(),0
fixed_noise = torch.randn(1, nz, 1, 1, device=device)

optimizer = optim.Adam(FSC_net.parameters(), lr=lr, betas=(beta1, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [8]:
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')
for i in fsc_submission.index:
    img=fsc_test[i].reshape((1, 3, 455, 256)).float().to(device)
    output = torch.argmax(FSC_net(img),dim=1).cpu().detach().numpy()
    fsc_submission.loc[i]['label']=output

fsc_submission=pandas.DataFrame(fsc_submission)
fsc_submission['image']=fsc_submission.index
fsc_submission.to_csv(f"submission.csv", index=False)
fsc_submission

/tmp/ipykernel_23/781579219.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  fsc_submission.loc[i]['label']=output


,label,image
image,,
102.jpg,0,102.jpg
108.jpg,1,108.jpg
109.jpg,1,109.jpg
111.jpg,1,111.jpg
121.jpg,1,121.jpg
...,...,...
899.jpg,0,899.jpg
91.jpg,0,91.jpg
94.jpg,0,94.jpg
